# Tutorial Debug — Per-Tier Training & Analysis

Train a DQN agent on **individual tutorial tiers** (up to 75 epochs each,
early-stop on mastery), then run each scenario with a greedy policy for
detailed move-by-move debugging.

**Flow per tier:**
1. Train agent on that tier's scenarios (75 epochs max, early stop at 90% mastery)
2. Switch to greedy policy (epsilon=0, noise off)
3. Run each scenario once with structured per-move logging
4. Restore exploration for next tier

Works with any agent variant (Baseline, NoisyNet, Dueling, IQN, Kitchen Sink, etc.).
Tiers are trained cumulatively — the agent carries over knowledge from prior tiers.

In [1]:
import sys, os, time, random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, deque

# Ensure project root is on path
_cwd = os.getcwd()
if os.path.basename(_cwd) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
elif os.path.isdir(os.path.join(_cwd, "simulation")):
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Project root: /home/franko/CT-DRL-MA
PyTorch: 2.10.0+cu128
CUDA: True


In [4]:
# ================================================================
# Configuration — change these to select variant and training params
# ================================================================

VARIANT = "spectral_norm"            # "baseline", "noisynet", "dueling", "iqn", "qrdqn",
                                # "munchausen", "spectralnorm", "kitchen_sink"
BACKBONE = "wider"           # "baseline", "wider", "deeper", "residual",
                                # "narrow_deep", "kitchen_sink"
MAX_EPOCHS_PER_TIER = 150        # Max training epochs per tier
MASTERY_THRESHOLD = 0.9         # 90% pass rate to consider mastered
WINDOW_SIZE = 20                # Rolling window for pass rate
MIN_EPOCHS = 10                 # Min epochs before mastery check
LOG_EVERY = 10                  # Print progress every N epochs

print(f"Variant:    {VARIANT}")
print(f"Backbone:   {BACKBONE}")
print(f"Max epochs: {MAX_EPOCHS_PER_TIER} per tier")
print(f"Mastery:    {MASTERY_THRESHOLD:.0%} over window of {WINDOW_SIZE}")

Variant:    spectral_norm
Backbone:   wider
Max epochs: 150 per tier
Mastery:    90% over window of 20


In [5]:
from simulation.rl.agent_registry import create_agent
from simulation.training.curriculum_trainer import (
    create_env_factory, build_agent_config,
)
from simulation.training.scenarios import (
    ALL_SCENARIOS, SCENARIO_BY_ID, TIER_DEFS,
    ONE_SHOT_SCENARIOS, REPEATABLE_SCENARIOS,
    TutorialResult, MoveRecord, StepEvent,
    visualize_scenario,
)
from simulation.training.tutorial_runner import (
    TutorialRunner, TUTORIAL_TIERS,
    TUTORIAL_REWARD_SUCCESS, TUTORIAL_REWARD_TIMEOUT,
)

# ---- Environment & Agent ----
env_factory = create_env_factory(rows=5, bays=58, tiers=5, tracks=7, split_factor=20)
base_cfg = build_agent_config(rows=5, bays=58, tiers=5, split_factor=20, tracks=7)

agent = create_agent(VARIANT, base_cfg, backbone_variant=BACKBONE)

n_params = sum(p.numel() for p in agent.q_net.parameters())
print(f"{VARIANT} agent created (from scratch)")
print(f"  Parameters: {n_params:,}")
print(f"  Backbone:   {type(agent.q_net.backbone).__name__}")
print(f"  Device:     {base_cfg.device}")
print(f"  Scenarios:  {len(ALL_SCENARIOS)} total "
      f"({len(ONE_SHOT_SCENARIOS)} one-shot, {len(REPEATABLE_SCENARIOS)} repeatable)")
print(f"  Tiers:      {len(TUTORIAL_TIERS)}")

runner = TutorialRunner(
    env_factory=env_factory,
    agent_or_config=agent,
    verbose=False,
)

# Global stores for cross-tier summary
ALL_RESULTS: dict[int, TutorialResult] = {}
TIER_TRAINING: dict[int, dict] = {}

print("Runner ready")

spectral_norm agent created (from scratch)
  Parameters: 557,443
  Backbone:   WiderCNNBackbone
  Device:     cuda
  Scenarios:  33 total (11 one-shot, 22 repeatable)
  Tiers:      14
Runner ready


/home/franko/.cache/pypoetry/virtualenvs/ct-drl-ma-7RJlkqYG-py3.12/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator KernelDensity from version 1.5.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# ================================================================
# Training, debug, and display functions
# ================================================================

try:
    from simulation.rl.variants.networks.noisy_linear import NoisyLinear
    _HAS_NOISY = True
except ImportError:
    _HAS_NOISY = False


def _has_noisy_layers() -> bool:
    """Check whether the current agent uses NoisyLinear layers."""
    if not _HAS_NOISY:
        return False
    return any(isinstance(m, NoisyLinear) for m in agent.q_net.modules())


def train_single_tier(
    tier_idx: int,
    max_epochs: int = MAX_EPOCHS_PER_TIER,
    mastery_threshold: float = MASTERY_THRESHOLD,
    window_size: int = WINDOW_SIZE,
    min_epochs: int = MIN_EPOCHS,
    log_every: int = LOG_EVERY,
) -> dict:
    """Train agent on a single tier. Stops early if mastered."""
    tier = TUTORIAL_TIERS[tier_idx]
    history = {s.id: deque(maxlen=window_size) for s in tier.scenarios}

    t0 = time.time()
    mastered = False
    epoch = 0

    sep = "=" * 70
    print()
    print(sep)
    print(f"  Training Tier {tier_idx}: {tier.name}")
    print(f"  Scenarios: {', '.join(f'S{s.id}' for s in tier.scenarios)}")
    print(f"  Max epochs: {max_epochs}  |  Mastery: {mastery_threshold:.0%}")
    print(sep)

    for epoch in range(max_epochs):
        agent.set_tutorial_epsilon(epoch)
        agent.set_tutorial_noise(epoch)          # anneal NoisyNet (no-op for others)

        # Run all scenarios in this tier
        steps = 0
        for sc in tier.scenarios:
            result = runner.run_scenario(sc)
            history[sc.id].append(1 if result.passed else 0)
            steps += result.steps

        # Optimize from replay buffer
        n_optim = min(10, max(1, steps // 4))
        for _ in range(n_optim):
            agent.optimize()

        # Compute pass rates
        pass_rates = {}
        for sid, hist in history.items():
            pass_rates[sid] = sum(hist) / len(hist) if len(hist) > 0 else 0.0
        avg_rate = np.mean(list(pass_rates.values()))

        # Log progress
        if (epoch + 1) % log_every == 0 or epoch == 0:
            rates_str = "  ".join(f"S{sid}={r:.0%}" for sid, r in sorted(pass_rates.items()))
            print(f"  Epoch {epoch+1:>3d}/{max_epochs}  avg={avg_rate:.0%}  {rates_str}")

        # Mastery check
        if epoch + 1 >= min_epochs and epoch + 1 >= window_size:
            if all(r >= mastery_threshold for r in pass_rates.values()):
                mastered = True
                rates_str = "  ".join(f"S{sid}={r:.0%}" for sid, r in sorted(pass_rates.items()))
                print(f"  Epoch {epoch+1:>3d}/{max_epochs}  avg={avg_rate:.0%}  {rates_str}")
                print(f"  >>> MASTERED at epoch {epoch+1}")
                break

    wall = time.time() - t0
    status = "MASTERED" if mastered else "NOT MASTERED"
    print()
    print(f"  Result: {status} in {epoch+1} epochs ({wall:.1f}s)")
    print(sep)

    return {
        "tier": tier_idx,
        "tier_name": tier.name,
        "epochs": epoch + 1,
        "mastered": mastered,
        "pass_rates": pass_rates,
        "wall_time": wall,
    }


def debug_tier(tier_idx: int) -> list[TutorialResult]:
    """Run each scenario once with greedy policy and return detailed results."""
    # ---- Enter greedy mode ----
    saved_override = agent.epsilon_override
    agent.epsilon_override = 0.0

    # For NoisyNet variants: silence noise via set_noise_scale
    is_noisy = _has_noisy_layers()
    if is_noisy:
        agent.set_noise_scale(0.0)

    # ---- Run scenarios ----
    tier = TUTORIAL_TIERS[tier_idx]
    results = []
    for sc in tier.scenarios:
        result = runner.run_scenario(sc)
        results.append(result)
        ALL_RESULTS[sc.id] = result

    # ---- Restore exploration ----
    agent.epsilon_override = saved_override
    if is_noisy:
        agent.set_noise_scale(1.0)

    # ---- Display ----
    display_tier_summary(f"Tier {tier_idx}: {tier.name}", results)
    for r in results:
        display_scenario_result(r, show_moves=True)

    return results


def display_scenario_result(result: TutorialResult, show_moves: bool = True):
    """Print structured per-move log for one scenario result."""
    status = "PASS" if result.passed else "FAIL"
    repeatable = SCENARIO_BY_ID[result.scenario_id].repeatable
    tag = "repeatable" if repeatable else "one-shot"

    sep = "=" * 90
    print()
    print(sep)
    print(f"S{result.scenario_id}: {result.name}  [{status}]  ({tag})")
    print(f"  total_reward={result.total_reward:+.4f}  "
          f"steps={result.steps}  moves={result.agent_moves}")
    if result.move_type_counts:
        types_str = ", ".join(f"{k}={v}" for k, v in sorted(result.move_type_counts.items()))
        print(f"  move types: {types_str}")
    if result.reward_breakdown:
        bd_str = ", ".join(f"{k}={v:+.2f}" for k, v in sorted(result.reward_breakdown.items()))
        print(f"  reward breakdown: {bd_str}")
    print(sep)

    if not show_moves:
        return

    if not result.move_log and not result.step_events:
        print("  (no moves or events)")
        return

    # Build timeline: moves + events sorted by step
    timeline = []
    for m in result.move_log:
        timeline.append((m.step, 1, "move", m))
    for e in result.step_events:
        timeline.append((e.step, 0, "event", e))
    timeline.sort(key=lambda x: (x[0], x[1]))

    hdr = "  {:>3s}  {:>4s}  {:>6s}  {:<28s}  {:<14s}  {:>7s}  {:>7s}  {:>9s}  {}"
    print(hdr.format("#", "Step", "Kind", "Move Type", "ID", "Dist", "Time", "Reward", "Notes"))
    print("  {}  {}  {}  {}  {}  {}  {}  {}  {}".format(
        "-"*3, "-"*4, "-"*6, "-"*28, "-"*14, "-"*7, "-"*7, "-"*9, "-"*20))

    for step_val, _, kind, data in timeline:
        if kind == "move":
            m = data
            flags = []
            productive = {"TRAIN_TO_YARD", "TRUCK_TO_YARD", "YARD_TO_TRAIN",
                          "YARD_TO_TRUCK", "TRAIN_TO_TRUCK", "TRUCK_TO_TRAIN",
                          "YARD_TO_TERMINAL_TRUCK"}
            if m.move_type in productive and m.reward < 0:
                flags.append("NEG_PROD")
            if m.move_type == "YARD_TO_YARD" and m.reward > 0:
                flags.append("POS_RESTACK")
            if abs(m.reward) > 15:
                flags.append("LARGE_R")
            if m.proximity_bonus and m.proximity_bonus > 0:
                flags.append(f"prox={m.proximity_bonus:.2f}")
            if m.src_bay is not None and m.dst_bay is not None:
                flags.append(f"r{m.src_row}b{m.src_bay}->r{m.dst_row}b{m.dst_bay}")

            flag_str = "  ".join(flags)
            print(f"  {m.move_num:>3d}  {step_val:>4d}  {'move':>6s}  "
                  f"{m.move_type:<28s}  {m.container_id:<14s}  "
                  f"{m.distance_m:>7.1f}  {m.time_s:>7.1f}  "
                  f"{m.reward:>+9.4f}  {flag_str}")
        else:
            e = data
            print(f"  {'':>3s}  {step_val:>4d}  {'event':>6s}  "
                  f"{e.event_type:<28s}  {'':14s}  {'':>7s}  {'':>7s}  "
                  f"{e.reward:>+9.4f}  {e.detail}")

    move_r = sum(m.reward for m in result.move_log)
    event_r = sum(e.reward for e in result.step_events)
    print()
    print(f"  Reward: moves={move_r:+.4f}  events={event_r:+.4f}  "
          f"total={result.total_reward:+.4f}")
    residual = result.total_reward - move_r - event_r
    if abs(residual) > 0.01:
        print(f"  *** RESIDUAL = {residual:+.4f} (unaccounted reward!) ***")


def display_tier_summary(tier_name: str, results: list[TutorialResult]):
    """Print a compact summary table for one tier."""
    n_pass = sum(1 for r in results if r.passed)
    line = chr(9472) * 60
    print()
    print(line)
    print(f"  {tier_name}: {n_pass}/{len(results)} passed")
    print(line)
    for r in results:
        sc = SCENARIO_BY_ID[r.scenario_id]
        tag = "one-shot" if not sc.repeatable else "repeat"
        status = "PASS" if r.passed else "FAIL"
        print(f"  [{status}] S{r.scenario_id:>2d} ({tag:>8s})  {r.name:<35s} "
              f"moves={r.agent_moves:>2d}  R={r.total_reward:>+8.2f}")


def train_and_debug_tier(tier_idx: int) -> list[TutorialResult]:
    """Train on a single tier, then debug with greedy policy."""
    summary = train_single_tier(tier_idx)
    TIER_TRAINING[tier_idx] = summary

    print()
    print("--- Greedy Debug (epsilon=0, noise off) ---")
    results = debug_tier(tier_idx)
    return results


print("Functions ready")

Functions ready


---
## Per-Tier Training & Debug

Each cell below trains the agent on one tier (up to 75 epochs, early-stop on mastery),
then runs a greedy debug pass with detailed move logs.

Run cells **sequentially** — the agent accumulates knowledge from prior tiers.
Or restart kernel and jump to a specific tier to test it in isolation.

### Tier 0: Primitives (S1-S4) — ONE-SHOT
Single-action: park truck, train import, yard-to-truck, yard-to-train.

In [7]:
tier_0_results = train_and_debug_tier(0)


  Training Tier 0: Primitives
  Scenarios: S1, S2, S3, S4
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=50%  S1=100%  S2=100%  S3=0%  S4=0%
  Epoch  10/150  avg=70%  S1=100%  S2=100%  S3=0%  S4=80%
  Epoch  20/150  avg=79%  S1=100%  S2=100%  S3=25%  S4=90%
  Epoch  30/150  avg=94%  S1=100%  S2=100%  S3=75%  S4=100%
  Epoch  33/150  avg=98%  S1=100%  S2=100%  S3=90%  S4=100%
  >>> MASTERED at epoch 33

  Result: MASTERED in 33 epochs (232.9s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 0: Primitives: 3/4 passed
────────────────────────────────────────────────────────────
  [PASS] S 1 (one-shot)  park_truck                          moves= 2  R=   +9.92
  [PASS] S 2 (one-shot)  train_import                        moves= 1  R=   +7.30
  [PASS] S 3 (one-shot)  yard_to_truck                       moves= 1  R=   +9.90
  [FAIL] S 4 (one-shot)  yard_to_train                       moves= 0  R=  -35.00

S1: park_truc

### Tier 1: Two-action chains (S5-S6, S26-S27) — ONE-SHOT
Import-then-export, park-then-import, and direct transfer chains (train-to-truck, truck-to-train).

In [8]:
tier_1_results = train_and_debug_tier(1)


  Training Tier 1: Two-action chains
  Scenarios: S5, S6, S26, S27
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=75%  S5=100%  S6=100%  S26=100%  S27=0%
  Epoch  10/150  avg=88%  S5=80%  S6=100%  S26=80%  S27=90%
  Epoch  20/150  avg=94%  S5=90%  S6=100%  S26=90%  S27=95%
  Epoch  20/150  avg=94%  S5=90%  S6=100%  S26=90%  S27=95%
  >>> MASTERED at epoch 20

  Result: MASTERED in 20 epochs (159.0s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 1: Two-action chains: 4/4 passed
────────────────────────────────────────────────────────────
  [PASS] S 5 (one-shot)  import_and_export                   moves= 2  R=  +16.50
  [PASS] S 6 (one-shot)  park_then_import                    moves= 2  R=   +7.79
  [PASS] S26 (one-shot)  train_to_truck_direct               moves= 3  R=  +11.45
  [PASS] S27 (one-shot)  truck_to_train_direct               moves= 2  R=  +15.54

S5: import_and_export  [PASS]  (one-shot)
  total_

### Tier 2: Full chains (S7-S8) — ONE-SHOT
Three-action chains: full truck chain, delivery-to-train chain.

In [9]:
tier_2_results = train_and_debug_tier(2)


  Training Tier 2: Full chains
  Scenarios: S7, S8
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=100%  S7=100%  S8=100%
  Epoch  10/150  avg=100%  S7=100%  S8=100%
  Epoch  20/150  avg=100%  S7=100%  S8=100%
  Epoch  20/150  avg=100%  S7=100%  S8=100%
  >>> MASTERED at epoch 20

  Result: MASTERED in 20 epochs (91.6s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 2: Full chains: 2/2 passed
────────────────────────────────────────────────────────────
  [PASS] S 7 (one-shot)  full_chain_truck                    moves= 3  R=  +11.42
  [PASS] S 8 (one-shot)  delivery_to_train                   moves= 2  R=  +16.24

S7: full_chain_truck  [PASS]  (one-shot)
  total_reward=+11.4181  steps=2  moves=3
  move types: PARK_TRUCK=1, TRAIN_TO_YARD=1, YARD_TO_TRUCK=1
  reward breakdown: PARK_TRUCK=+0.10, SUCCESS_BONUS=+5.00, TRAIN_TO_YARD=+2.29, YARD_TO_TRUCK=+2.03, non_move=+2.00
    #  Step    Kind  Move Type            

### Tier 3: Restack + load (S9-S10) — REPEATABLE
Unbury containers with distractors, randomised layouts.

In [10]:
tier_3_results = train_and_debug_tier(3)


  Training Tier 3: Restack + load
  Scenarios: S9, S10
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=0%  S9=0%  S10=0%
  Epoch  10/150  avg=65%  S9=60%  S10=70%
  Epoch  20/150  avg=70%  S9=65%  S10=75%
  Epoch  30/150  avg=82%  S9=85%  S10=80%
  Epoch  40/150  avg=82%  S9=95%  S10=70%
  Epoch  50/150  avg=88%  S9=95%  S10=80%
  Epoch  55/150  avg=95%  S9=100%  S10=90%
  >>> MASTERED at epoch 55

  Result: MASTERED in 55 epochs (576.0s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 3: Restack + load: 2/2 passed
────────────────────────────────────────────────────────────
  [PASS] S 9 (  repeat)  unbury_for_truck                    moves= 2  R=   +9.43
  [PASS] S10 (  repeat)  unbury_for_train                    moves= 2  R=  +13.86

S9: unbury_for_truck  [PASS]  (repeatable)
  total_reward=+9.4308  steps=1  moves=2
  move types: YARD_TO_TRUCK=1, YARD_TO_YARD=1
  reward breakdown: SUCCESS_BONUS=+5.00, YARD_TO

### Tier 4: Random generalize (S11-S12) — REPEATABLE
Random import/export chains with randomised container types and positions.

In [12]:
tier_4_results = train_and_debug_tier(4)


  Training Tier 4: Random generalize
  Scenarios: S11, S12
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=100%  S11=100%  S12=100%
  Epoch  10/150  avg=95%  S11=90%  S12=100%
  Epoch  20/150  avg=95%  S11=95%  S12=95%
  Epoch  20/150  avg=95%  S11=95%  S12=95%
  >>> MASTERED at epoch 20

  Result: MASTERED in 20 epochs (219.0s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 4: Random generalize: 2/2 passed
────────────────────────────────────────────────────────────
  [PASS] S11 (  repeat)  random_import_chain                 moves= 5  R=  +10.35
  [PASS] S12 (  repeat)  random_export_chain                 moves= 3  R=  +14.59

S11: random_import_chain  [PASS]  (repeatable)
  total_reward=+10.3501  steps=2  moves=5
  move types: PARK_TRUCK=1, TRAIN_TO_YARD=1, YARD_TO_TRUCK=1, YARD_TO_YARD=2
  reward breakdown: PARK_TRUCK=+0.10, SUCCESS_BONUS=+5.00, TRAIN_TO_YARD=+2.29, YARD_TO_TRUCK=+2.03, YARD_TO_YARD=-1.07, 

### Tier 5: Multi-step hard (S13-S14) — REPEATABLE
Multi-truck serving and deep unbury scenarios.

In [13]:
tier_5_results = train_and_debug_tier(5)


  Training Tier 5: Multi-step hard
  Scenarios: S13, S14
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=50%  S13=100%  S14=0%
  Epoch  10/150  avg=80%  S13=100%  S14=60%
  Epoch  20/150  avg=82%  S13=100%  S14=65%
  Epoch  30/150  avg=92%  S13=100%  S14=85%
  Epoch  31/150  avg=95%  S13=100%  S14=90%
  >>> MASTERED at epoch 31

  Result: MASTERED in 31 epochs (338.8s)

--- Greedy Debug (epsilon=0, noise off) ---

────────────────────────────────────────────────────────────
  Tier 5: Multi-step hard: 1/2 passed
────────────────────────────────────────────────────────────
  [PASS] S13 (  repeat)  multi_truck_serving                 moves= 3  R=  +19.70
  [FAIL] S14 (  repeat)  deep_unbury                         moves=60  R=  -27.61

S13: multi_truck_serving  [PASS]  (repeatable)
  total_reward=+19.7033  steps=2  moves=3
  move types: YARD_TO_TRUCK=3
  reward breakdown: SUCCESS_BONUS=+5.00, YARD_TO_TRUCK=+8.70, non_move=+6.00
    #  Step    Kind  Move Type                     I

### Tier 6: Terminal truck (S25, S22-S24) — MIXED
S25 (basic dispatch) is one-shot; S22-S24 are repeatable.

In [ ]:
tier_6_results = train_and_debug_tier(6)


  Training Tier 6: Terminal truck
  Scenarios: S25, S22, S23, S24
  Max epochs: 150  |  Mastery: 90%
  Epoch   1/150  avg=75%  S22=100%  S23=100%  S24=100%  S25=0%


### Tier 7: Multi-vehicle (S15-S16) — REPEATABLE
Multiple delivery trucks and delivery-to-train pipeline.

In [ ]:
tier_7_results = train_and_debug_tier(7)

### Tier 8: Bidirectional train (S17) — REPEATABLE
Simultaneous import and export on the same train.

In [ ]:
tier_8_results = train_and_debug_tier(8)

### Tier 9: Concurrent ops (S18-S19) — REPEATABLE
Concurrent import+export and mixed fleet dispatch.

In [ ]:
tier_9_results = train_and_debug_tier(9)

### Tier 10: Full complexity (S20-S21) — REPEATABLE
Crowded yard pickup and full mini-day stress tests.

In [ ]:
tier_10_results = train_and_debug_tier(10)

### Tier 11: Full train ops (S28-S29) — REPEATABLE
Full train unload (6 imports to yard) and full train load (6 exports from yard, 3 buried under blockers).

In [ ]:
tier_11_results = train_and_debug_tier(11)

### Tier 12: Cross-modal orchestration (S30-S31) — REPEATABLE
Multi-truck to train coordination and full bidirectional day (capstone benchmark with every move type).

In [ ]:
tier_12_results = train_and_debug_tier(12)

In [ ]:
tier_13_results = train_and_debug_tier(13)

---
## Global Summary

In [ ]:
# ================================================================
# Global summary DataFrame (from ALL_RESULTS)
# ================================================================

if not ALL_RESULTS:
    print("No results yet -- run the tier cells above first!")
else:
    rows = []
    for sid in sorted(ALL_RESULTS.keys()):
        r = ALL_RESULTS[sid]
        sc = SCENARIO_BY_ID[sid]
        types_str = ", ".join(f"{k}={v}" for k, v in sorted(r.move_type_counts.items()))
        rows.append({
            "ID": f"S{sid}",
            "Name": r.name,
            "Type": "one-shot" if not sc.repeatable else "repeat",
            "Pass": "PASS" if r.passed else "FAIL",
            "Steps": r.steps,
            "Moves": r.agent_moves,
            "Total R": round(r.total_reward, 2),
            "Move Types": types_str,
        })

    df_summary = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.max_rows", 30)

    n_pass = sum(1 for r in ALL_RESULTS.values() if r.passed)
    n_total = len(ALL_RESULTS)
    print(f"Overall: {n_pass}/{n_total} passed ({n_pass/n_total*100:.0f}%)")
    print()
    display(df_summary)

    # Training summary
    if TIER_TRAINING:
        print()
        print("Training Summary:")
        print("  {:<6s}  {:<25s}  {:>7s}  {:<12s}  {:>8s}".format(
            "Tier", "Name", "Epochs", "Status", "Time"))
        print("  {}  {}  {}  {}  {}".format(
            "-"*6, "-"*25, "-"*7, "-"*12, "-"*8))
        for tidx in sorted(TIER_TRAINING.keys()):
            t = TIER_TRAINING[tidx]
            status = "MASTERED" if t["mastered"] else "not mastered"
            print(f"  {tidx:<6d}  {t['tier_name']:<25s}  {t['epochs']:>7d}  {status:<12s}  {t['wall_time']:>7.1f}s")

## Reward Distribution Charts

In [ ]:
# ================================================================
# Reward distribution analysis
# ================================================================

if not ALL_RESULTS:
    print("No results yet -- run the tier cells above first!")
else:
    all_moves = []
    for sid, r in ALL_RESULTS.items():
        for m in r.move_log:
            all_moves.append({
                "scenario": f"S{sid}",
                "move_type": m.move_type,
                "reward": m.reward,
                "distance_m": m.distance_m,
                "time_s": m.time_s,
            })

    df_moves = pd.DataFrame(all_moves)
    print(f"Total moves across {len(ALL_RESULTS)} scenarios: {len(df_moves)}")
    print()

    if len(df_moves) > 0:
        print(df_moves.groupby("move_type").agg(
            count=("reward", "count"),
            mean_reward=("reward", "mean"),
            min_reward=("reward", "min"),
            max_reward=("reward", "max"),
            mean_dist=("distance_m", "mean"),
            mean_time=("time_s", "mean"),
        ).round(4).to_string())

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # 1) Average reward by move type
        ax = axes[0]
        grouped = df_moves.groupby("move_type")["reward"].mean().sort_values()
        colors = ["#e74c3c" if v < 0 else "#27ae60" for v in grouped.values]
        grouped.plot.barh(ax=ax, color=colors, edgecolor="white")
        ax.set_xlabel("Mean Reward")
        ax.set_title("Average Reward by Move Type")
        ax.axvline(0, color="gray", lw=0.5)

        # 2) Reward histogram
        ax = axes[1]
        for mt in sorted(df_moves["move_type"].unique()):
            subset = df_moves[df_moves["move_type"] == mt]["reward"].dropna()
            if len(subset) > 0:
                ax.hist(subset, bins=20, alpha=0.6, label=mt)
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.set_title("Reward Distribution by Move Type")
        ax.legend(fontsize=7, loc="upper left")

        # 3) Distance vs reward
        ax = axes[2]
        container_moves = df_moves[df_moves["distance_m"] > 0].copy()
        for mt in sorted(container_moves["move_type"].unique()):
            subset = container_moves[container_moves["move_type"] == mt]
            ax.scatter(subset["distance_m"], subset["reward"], alpha=0.6, label=mt, s=30)
        ax.set_xlabel("Distance (m)")
        ax.set_ylabel("Reward")
        ax.set_title("Distance vs Reward (container moves)")
        ax.legend(fontsize=7)

        plt.tight_layout()
        plt.show()

## Save Checkpoint

Save the trained agent in the format expected by `final_eval.ipynb`
(`runs/tutorial_checkpoints/{variant}_{backbone}_best.pt`).

In [ ]:
from pathlib import Path

CKPT_DIR = Path(PROJECT_ROOT) / 'runs' / 'tutorial_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

label = f"{VARIANT}_{BACKBONE}"
ckpt_path = CKPT_DIR / f"{label}_best.pt"

agent.save(str(ckpt_path))
print(f"Saved checkpoint: {ckpt_path}")
print(f"  step_count = {agent.step_count}")
print(f"  Load in final_eval.ipynb with agent label '{label}'")